# Correcci?n del score de selecci?n adversa

Este notebook documenta una correcci?n metodol?gica importante. Se hab?a usado un score relacionado con selecci?n adversa con la orientaci?n equivocada: seleccionar `P(adverse)` alto equivale a priorizar situaciones peligrosas, no sanas.

La correcci?n forma parte de la trazabilidad del proyecto: no se borra el error, se declara y se invalida la conclusi?n afectada.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

key = pd.read_csv(ROOT / 'results' / 'key_results.csv')
summary = pd.read_csv(ROOT / 'results' / 'final_candidate_summary.csv')
ledger = pd.read_csv(ROOT / 'results' / 'final_candidate_actions_anonymized.csv')

## Qu? se corrigi?

El score operativo correcto es `safe_adverse_score = 1 - P(adverse)`. Tras corregirlo, la antigua candidata basada en `P(adverse)` deja de ser evidencia v?lida.

In [2]:
pd.DataFrame([
    {"elemento": "Target adverso", "interpretaci?n correcta": "1 = peor cuartil de ejecuci?n / mayor riesgo"},
    {"elemento": "Score raw P(adverse)", "interpretaci?n correcta": "Alto es peor, no mejor"},
    {"elemento": "Score safe", "interpretaci?n correcta": "1 - P(adverse), alto es m?s sano"},
    {"elemento": "Consecuencia", "interpretaci?n correcta": "Las pol?ticas raw adversas quedan invalidadas"},
])

,elemento,interpretaci?n correcta
0,Target adverso,1 = peor cuartil de ejecuci?n / mayor riesgo
1,Score raw P(adverse),"Alto es peor, no mejor"
2,Score safe,"1 - P(adverse), alto es m?s sano"
3,Consecuencia,Las pol?ticas raw adversas quedan invalidadas


## Evidencia resumida en el registro p?blico

El registro de resultados conserva la decisi?n corregida: no hay configuraciones robustas basadas en `safe_adverse` que pasen el protocolo.

In [3]:
mask = key["phase"].isin(["methodology", "oof"]) | key["experiment"].str.contains("nested|fusion", case=False, na=False)
key.loc[mask, ["phase", "experiment", "metric", "value", "unit", "status", "interpretation"]]

,phase,experiment,metric,value,unit,status,interpretation
8,methodology,nested_v3_corrected,safe_adverse_robust_configs,0,count,NO_GO,Earlier adverse candidate invalidated
9,oof,clean_h60_upstream,test_auc,0.5656,auc,PARTIAL_SIGNAL,Modest causal ranking signal
10,oof,clean_h60_upstream,test_top35_cost0p50_mean,0.1778,proxy_ticks,PARTIAL_SIGNAL,Positive under primary proxy cost
11,oof,clean_h60_upstream,test_top35_cost1p00_mean,-0.1899,proxy_ticks,NO_GO,Does not survive conservative cost
12,oof,clean_fusion_stage2,robust_configs,0,count,NO_GO,Second stage does not create stable policy


## Lectura

Esta correcci?n refuerza el mensaje metodol?gico de la memoria: no basta con que una m?trica sea positiva; hay que auditar qu? significa econ?micamente. Despu?s de corregir la orientaci?n, la pista no sobrevive como pol?tica seleccionable.